In [1]:
import mlflow
import mlflow.xgboost
import joblib
from sklearn.metrics import roc_auc_score, precision_score, recall_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("fraud-detection")

train_df = joblib.load('../data/processed/train_processed_v2.pkl')
val_df = joblib.load('../data/processed/val_processed_v2.pkl')

model_v1 = joblib.load('../models/xgb_fraud_model_v1.pkl')
feature_cols = model_v1.get_booster().feature_names

X_val = val_df[feature_cols]
y_val = val_df['isFraud']

proba = model_v1.predict_proba(X_val)[:, 1]
pred = model_v1.predict(X_val)

auc = roc_auc_score(y_val, proba)
precision = precision_score(y_val, pred)
recall = recall_score(y_val, pred)

print(f"AUC: {auc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")

2026/09/29 18:12:14 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection' does not exist. Creating a new experiment.


AUC: 0.9066, Precision: 0.2263, Recall: 0.7207


In [2]:
with mlflow.start_run(run_name="xgb_v1_baseline"):
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("features_used", "velocity_features_v2")
    
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    
    mlflow.xgboost.log_model(model_v1, name="model", registered_model_name="fraud-detection-model")
    
    print("Logged run successfully.")

Successfully registered model 'fraud-detection-model'.
2026/09/29 18:12:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud-detection-model, version 1
Created version '1' of model 'fraud-detection-model'.


Logged run successfully.
🏃 View run xgb_v1_baseline at: http://localhost:5000/#/experiments/1/runs/0e1da8400e2444f9b9abd9d031707c6a
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [3]:
retrained_model = joblib.load('../models/xgb_fraud_model_v2_retrained.pkl')

with open('../data/drift_simulation/drift_metadata.json') as f:
    import json
    drift_meta = json.load(f)

with mlflow.start_run(run_name="xgb_v2_retrained_post_drift"):
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("trigger_reason", "drift_detected")
    mlflow.log_param("retrain_set_size", 41337)
    
    mlflow.log_metric("auc_roc_holdout", 0.9441)
    mlflow.log_metric("precision_holdout", 0.5034)
    mlflow.log_metric("recall_holdout", 0.7890)
    mlflow.log_metric("old_model_auc_on_drift", 0.7007)
    mlflow.log_metric("improvement_over_old", 0.2434)
    
    mlflow.xgboost.log_model(retrained_model, name="model", registered_model_name="fraud-detection-model")
    
    print("Logged retrained model successfully.")

Registered model 'fraud-detection-model' already exists. Creating a new version of this model...
2026/09/29 18:13:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud-detection-model, version 2
Created version '2' of model 'fraud-detection-model'.


Logged retrained model successfully.
🏃 View run xgb_v2_retrained_post_drift at: http://localhost:5000/#/experiments/1/runs/31bfb988f53a48c48e5a41efa3956b24
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [4]:
from mlflow import MlflowClient

client = MlflowClient()

# Set version 2 as the "champion" (production) model, since it was validated and promoted
client.set_registered_model_alias("fraud-detection-model", "champion", 2)

# Keep version 1 tagged as the prior baseline for reference
client.set_registered_model_alias("fraud-detection-model", "previous", 1)

print("Aliases set.")

# Verify
model_info = client.get_model_version_by_alias("fraud-detection-model", "champion")
print(f"Champion model: version {model_info.version}, run_id {model_info.run_id}")

Aliases set.
Champion model: version 2, run_id 31bfb988f53a48c48e5a41efa3956b24
